<a href="https://colab.research.google.com/github/ShubhendraP/AgenticAI2026/blob/weekly-classes/crewai_multiagent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# CrewAI — Multi-Agent Framework

## What is CrewAI?

**CrewAI** is an open-source framework for building multi-agent systems where each agent plays a **specialised role**, just like employees in a company.

Instead of one large LLM doing everything, CrewAI breaks the work into **specialist agents** who collaborate:

| Agent Component | What it means |
|---|---|
| **Role** | What the agent *is* (e.g., "Content Planner") |
| **Goal** | What the agent *wants to accomplish* |
| **Backstory** | Context that shapes the agent's behaviour and tone |
| **Task** | The specific work assigned to this agent |
| **Tools** | Optional external tools the agent can call |

### Real-World Analogy: A Magazine Editorial Team
- The **Editor-in-Chief** (Planner) decides what topics to cover and outlines the structure
- The **Journalist** (Writer) writes the article based on the plan
- The **Copy Editor** (Editor) reviews, polishes, and ensures brand consistency

Each person has a different role. None of them do each other's job. They pass work sequentially.

### Industry Examples of CrewAI-style Teams:
- **Software development**: Architect agent → Developer agent → Code reviewer agent
- **Market research**: Research agent → Analyst agent → Report writer agent
- **Customer support**: Triage agent → Specialist agent → Follow-up agent

### How CrewAI differs from a single LLM:
| Aspect | Single LLM | CrewAI Multi-Agent |
|---|---|---|
| Focus | Generalised | Specialised roles |
| Quality | One pass | Multiple review stages |
| Scalability | Limited | Add more agents easily |
| Real-world fit | Simple tasks | Complex, multi-stage workflows |

---
### Our Scenario: AI Content Creation Team
We build a 3-agent content pipeline:
- **Planner** → researches the topic and creates an outline
- **Writer** → turns the outline into a full article
- **Editor** → polishes the article for publication

Watch how each agent builds on the previous one's work.

# Agents to Research and Write an Article

Foundational concepts of multi-agent systems and get an overview of the crewAI framework.

For running this notebook on your own machine, you can install the following:
```Python
!pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29
```

In [1]:
 !pip install crewai==0.28.8 crewai_tools==0.1.6


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of embedchain to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of embedchain to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
INFO: pip is looking at multiple versions of embedchain[github,youtube] to determine which version is compatible with other requirements. This could take a while.


In [1]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

- Import from the crewAI libray.

In [2]:
from crewai import Agent, Task, Crew

In [3]:
import os
from langchain_openai import ChatOpenAI
from google.colab import userdata


os.environ['OPENAI_API_KEY']   = userdata.get('OPENAI_API_KEY')
os.environ['OPENAI_BASE_URL']  = 'https://openai.vocareum.com/v1'

import warnings
warnings.filterwarnings('ignore')
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

## Creating Agents

- Define your Agents, and provide them a `role`, `goal` and `backstory`.
- It has been seen that LLMs perform better when they are role playing.

### Agent: Planner


In [4]:
planner = Agent(
    role="Content Planner",
    goal="Plan engaging and factually accurate content on {topic}",
    backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
    llm=llm,
	verbose=True
)

### Agent: Writer

In [5]:
writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    llm=llm,
    verbose=True
)

### Agent: Editor

In [6]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    llm=llm,
    verbose=True
)

## Creating Tasks

- Define your Tasks, and provide them a `description`, `expected_output` and `agent`.

### Task: Plan

In [7]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner,
)

### Task: Write

In [8]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)

### Task: Edit

In [9]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 1 or 2 paragraphs.",
    agent=editor
)

## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution.

In [10]:
# ─────────────────────────────────────────────────────────────
# CREATING THE CREW: Assembling the multi-agent team
#
# A Crew ties together:
#   - agents:  the team members
#   - tasks:   the work assigned to each member
#   - process: how the work flows (sequential = one after another)
#
# KEY INSIGHT: The ORDER of tasks in the list matters in sequential mode.
# plan → write → edit  means the planner always goes first.
# The writer can only start after the planner finishes.
# The editor can only start after the writer finishes.
#
# This mirrors a real editorial pipeline — you can't edit before writing!
# ─────────────────────────────────────────────────────────────
crew = Crew(
    agents=[planner, writer, editor],  # The team
    tasks=[plan, write, edit],         # Work in this exact order
    verbose=2                          # Show detailed logs of each agent's reasoning
)

## Running the Crew

**Note**: LLMs can provide different outputs for they same input, so what you get might be different than what you see here.

In [11]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer.  
Final Answer: 

**Comprehensive Content Plan for Blog Article on Artificial Intelligence**

---

### 1. Latest Trends, Key Players, and Noteworthy News in Artificial Intelligence

**Trends:**
- **Generative AI:** The rise of AI models that can create text, images, and music, such as OpenAI's ChatGPT and DALL-E.
- **AI Ethics and Regulation:** Increasing focus on ethical AI use, data privacy, and the need for regulatory frameworks.
- **AI in Healthcare:** Advancements in AI applications for diagnostics, personalized medici

- Display the results of your execution as markdown in the notebook.

In [12]:
from IPython.display import Markdown
Markdown(result)

```markdown
# Understanding Artificial Intelligence: Trends, Impacts, and Future Directions

## I. Introduction

Artificial Intelligence (AI) has rapidly evolved from a niche area of research to a transformative force across various sectors. Its significance in today's world cannot be overstated, as AI technologies are reshaping industries, enhancing productivity, and influencing our daily lives. As we navigate this technological landscape, it is crucial to stay informed about the latest trends and developments in AI, which promise to revolutionize how we work while raising important ethical and regulatory questions.

In this article, we will explore the latest trends in AI, identify key players in the field, discuss ethical considerations, and predict future directions for this dynamic technology. Whether you are a tech enthusiast, a business professional, or simply curious about AI's impact on society, this overview aims to provide valuable insights into the current state and future of artificial intelligence.

## II. Latest Trends in AI

### Generative AI and Its Applications

One of the most exciting trends in AI is the rise of generative AI, which refers to models capable of creating text, images, and even music. Notable examples include OpenAI's ChatGPT and DALL-E, which have garnered significant attention for their ability to produce human-like text and stunning visuals. These advancements are not just technological marvels; they have practical applications in content creation, marketing, and entertainment, making them invaluable tools for businesses and creators alike.

Moreover, generative AI is pushing the boundaries of creativity, allowing users to generate unique content tailored to specific needs. As these technologies continue to evolve, we can expect to see even more innovative applications that will redefine how we interact with digital media.

### The Role of AI in Healthcare

AI's impact on healthcare is another area of significant growth. From diagnostics to personalized medicine, AI technologies are enhancing patient care and improving outcomes. For instance, AI algorithms can analyze medical images with remarkable accuracy, assisting radiologists in detecting conditions like cancer at earlier stages. Additionally, AI-driven tools are being used to develop personalized treatment plans based on individual patient data, leading to more effective therapies.

As the healthcare industry increasingly adopts AI solutions, the potential for improved patient outcomes and operational efficiencies becomes evident. However, this trend also raises questions about data privacy and the ethical use of sensitive health information, which we will explore further in the next section.

## III. Key Players in the AI Landscape

### Overview of Leading Companies and Their Contributions

The AI landscape is populated by several key players who are driving innovation and shaping the future of this technology. OpenAI stands out for its groundbreaking work in generative models, while Google DeepMind is renowned for its pioneering research in machine learning. IBM Watson has made significant strides in applying AI to business and healthcare, demonstrating the versatility of AI technologies across sectors.

Microsoft is also a major player, integrating AI into its products and services, including Azure AI, which provides businesses with powerful tools to harness AI capabilities. Additionally, NVIDIA has established itself as a leader in AI hardware and software solutions, enabling the development of advanced AI applications. Together, these companies are not only advancing AI technology but also setting the stage for future innovations.

### Discussion on Emerging Startups and Innovations

In addition to established giants, numerous startups are emerging with innovative AI solutions that address specific industry needs. These companies are often agile and able to pivot quickly, allowing them to explore niche markets and develop cutting-edge applications. As investment in AI startups continues to grow, we can expect a wave of new technologies that will further enhance the capabilities of AI and its integration into everyday life.

## IV. Ethical Considerations and Regulations

### Importance of Ethical AI Use

As AI technologies become more pervasive, the importance of ethical considerations cannot be overlooked. Issues such as data privacy, algorithmic bias, and the potential for job displacement are at the forefront of discussions surrounding AI. It is essential for developers and organizations to prioritize ethical AI use, ensuring that these technologies are designed and implemented in ways that are fair, transparent, and accountable.

The conversation around AI ethics is gaining momentum, with various stakeholders advocating for responsible practices that protect individuals and society as a whole. This includes developing guidelines for ethical AI use and fostering a culture of accountability among AI practitioners.

### Current Regulatory Landscape and Future Implications

The regulatory landscape for AI is still evolving, with governments and organizations worldwide recognizing the need for frameworks that govern AI technologies. Recent legislative developments indicate a growing commitment to establishing regulations that address the ethical use of AI, data privacy, and accountability. As these regulations take shape, they will play a crucial role in shaping the future of AI and ensuring that its benefits are realized without compromising ethical standards.

## V. Future Directions of AI

### Predictions for AI Advancements in the Next 5-10 Years

Looking ahead, the future of AI is poised for remarkable advancements. We can expect to see continued improvements in natural language processing (NLP), enabling even more sophisticated interactions between humans and machines. Additionally, the integration of AI into various sectors, including finance, education, and transportation, will likely lead to increased efficiency and innovation.

Moreover, as AI technologies become more accessible, we may witness a democratization of AI, allowing smaller businesses and individuals to leverage these tools for their own purposes. This shift could lead to a surge in creativity and innovation across industries, as more people harness the power of AI to solve problems and create new solutions.

### Potential Societal Impacts and Changes

The societal impacts of AI will be profound, influencing everything from job markets to daily life. While concerns about job displacement are valid, it is essential to recognize that AI also has the potential to create new job opportunities and enhance existing roles. As AI takes over repetitive tasks, human workers can focus on more complex and creative endeavors, leading to a shift in the nature of work.

Furthermore, AI's ability to analyze vast amounts of data can lead to more informed decision-making in various fields, from healthcare to environmental sustainability. As we embrace these changes, it is crucial to remain vigilant about the ethical implications and ensure that AI serves the greater good.

## VI. Conclusion

In summary, artificial intelligence is a rapidly evolving field that holds immense potential for transforming industries and society as a whole. From generative AI and healthcare advancements to the ethical considerations and regulatory frameworks shaping its future, staying informed about AI trends is essential for anyone interested in the implications of this technology.

As we move forward, it is vital to engage in discussions about the ethical use of AI and advocate for responsible practices that prioritize the well-being of individuals and communities. By doing so, we can harness the power of AI to create a brighter future for all.

## VII. Call to Action

If you found this article insightful, consider subscribing for updates on the latest AI trends and developments. Share your thoughts on AI in the comments below, and don't forget to share this article on social media to spread awareness about the importance of understanding artificial intelligence.
```

## Try it Yourself

- Pass in a topic of your choice and see what the agents come up with!

## Key Takeaways

| CrewAI Component | Purpose |
|---|---|
| `Agent(role, goal, backstory)` | Defines a specialist with a distinct persona |
| `Task(description, expected_output, agent)` | Assigns specific work to an agent |
| `Crew(agents, tasks, verbose)` | Orchestrates the team and runs the pipeline |
| Sequential process | Each task runs after the previous one completes |

### CrewAI vs. Custom Multi-Agent (multi_agent_system.ipynb):
| | CrewAI | Custom |
|---|---|---|
| Setup | Structured — use Agent(), Task(), Crew() | Flexible — any pattern you want |
| Debugging | verbose=2 shows detailed logs | You control the prints |
| Parallelism | Supports parallel tasks (advanced) | Manual |
| Production readiness | ✅ Higher | ❌ More work needed |

### The "role + goal + backstory" pattern:
This is why CrewAI agents often produce better results than a single LLM:
- **Role** sets the expertise context ("You are a Content Planner")
- **Goal** gives direction ("Plan engaging content")
- **Backstory** adds rich context that makes the LLM commit to the persona

Research shows LLMs perform significantly better when given a clear role to play.

### Try it yourself:
Change the `topic` in the last cell to anything — "Generative AI", "Climate Change", "Quantum Computing".
See how the 3-agent team produces a more structured result than asking a single LLM directly.

---
**Next steps:** Add `tools` to your agents (e.g., web search, file read) to make them even more powerful.
**See:** LangChain tools and CrewAI tool documentation.

In [ ]:
topic = "YOUR TOPIC HERE"
result = crew.kickoff(inputs={"topic": topic})

In [ ]:
Markdown(result)